# QualCheck – Training Pipeline (SciEntsBank Edition)

**Camarines Sur Polytechnic Colleges – College of Computer Studies**

This notebook adapts QualCheck to the **SciEntsBank** dataset ([Dzikovska et al., SemEval-2013](https://aclanthology.org/S13-2045)), loaded directly from HuggingFace.

### SciEntsBank → QualCheck mapping
| SciEntsBank column | Role in QualCheck |
|---|---|
| `question` | `question` |
| `student_answer` | `response` |
| `reference_answer` | `rubric_descriptor` (model answer = rubric) |
| 5-way label | mapped to 3-class QualCheck label (see Config) |

### Label mapping (5-way → 3-way QualCheck)
| SciEntsBank label | QualCheck label |
|---|---|
| `correct` | Fully Relevant |
| `contradictory`, `partially_correct_incomplete` | Partially Relevant |
| `irrelevant`, `non_domain` | Irrelevant |

### Test splits evaluated
SciEntsBank ships with **three** held-out test sets:
- `test_ua` – Unseen Answers (540 rows)
- `test_uq` – Unseen Questions (733 rows)
- `test_ud` – Unseen Domains (4,562 rows)

The notebook evaluates QualCheck on all three and reports results per split.

In [ ]:
!pip install transformers torch scikit-learn pandas numpy nltk spacy scipy statsmodels matplotlib seaborn tqdm datasets -q
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
print('✓ Packages ready')

In [ ]:
import os, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (BertTokenizer, BertModel,
                          get_linear_schedule_with_warmup)
from torch.optim import AdamW

from datasets import load_dataset    # HuggingFace datasets

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score,
                              precision_score, recall_score)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as tfidf_cos

from scipy.stats  import pearsonr, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import nltk
from nltk.corpus import stopwords
from nltk.stem   import WordNetLemmatizer

print('✓ Imports complete')


## Configuration

In [ ]:
class Config:
    # ── Dataset ───────────────────────────────────────────────────
    HF_DATASET   = "nkazi/SciEntsBank"   # HuggingFace dataset id

    # ── SciEntsBank 5-way → 3-way QualCheck mapping ───────────────
    # SciEntsBank integer labels (from HF):
    #   0 = correct
    #   1 = contradictory
    #   2 = partially_correct_incomplete
    #   3 = irrelevant
    #   4 = non_domain
    SEB_TO_QC = {
        0: "Fully Relevant",       # correct
        1: "Partially Relevant",   # contradictory
        2: "Partially Relevant",   # partially_correct_incomplete
        3: "Irrelevant",           # irrelevant
        4: "Irrelevant",           # non_domain
    }

    # ── QualCheck labels ──────────────────────────────────────────
    LABEL2ID   = {"Fully Relevant": 2, "Partially Relevant": 1, "Irrelevant": 0}
    ID2LABEL   = {2: "Fully Relevant",  1: "Partially Relevant", 0: "Irrelevant"}
    NUM_LABELS = 3

    # ── Rubric template ───────────────────────────────────────────
    # The reference_answer is prepended with this prefix before being
    # used as the rubric descriptor fed to BERT.
    RUBRIC_PREFIX = "A complete and correct answer should state: "

    # ── BERT ─────────────────────────────────────────────────────
    BERT_MODEL = "bert-base-uncased"
    MAX_LEN    = 128    # short answers → 128 is sufficient

    # ── Training ─────────────────────────────────────────────────
    BATCH_SIZE   = 32   # short texts allow larger batch
    LR           = 2e-5
    MAX_EPOCHS   = 5
    PATIENCE     = 3
    WARMUP_RATIO = 0.1
    DROPOUT      = 0.3

    # ── Split (train → 85% train / 15% val; all 3 test sets kept) ─
    VAL_RATIO  = 0.15
    SEED       = 42

    # ── Threshold calibration ────────────────────────────────────
    THRESH_STEP = 0.01

    # ── Output ───────────────────────────────────────────────────
    OUTPUT_DIR  = "qualcheck_seb_output"
    BEST_CKPT   = "qualcheck_seb_output/best_model.pt"
    FINAL_MODEL = "qualcheck_seb_output/qualcheck_seb_final.pt"

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

random.seed(cfg.SEED); np.random.seed(cfg.SEED); torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)

print(f"Device : {cfg.DEVICE}")
print(f"CUDA   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


## Phase 1 – Data Loading & Adaptation

### Step 1 – Load from HuggingFace

In [ ]:
print(f"Loading {cfg.HF_DATASET} from HuggingFace …")
raw = load_dataset(cfg.HF_DATASET)

print("\nAvailable splits:")
for name, ds in raw.items():
    print(f"  {name:<12}  {len(ds):>6,} rows  |  columns: {ds.column_names}")

# Preview one row
sample = raw["train"][0]
print("\nSample train row:")
for k, v in sample.items():
    print(f"  {k:<20}: {str(v)[:80]}")


### Step 2 – Convert SciEntsBank → QualCheck Format

In [ ]:
def convert_split(hf_dataset, cfg):
    """
    Maps SciEntsBank columns/labels to QualCheck format.
      question        → question
      reference_answer→ rubric_descriptor (prefixed)
      student_answer  → response
      label (int 0-4) → quality_label + label_id
    """
    rows = []
    for item in hf_dataset:
        raw_label = item["label"]
        qc_label  = cfg.SEB_TO_QC.get(raw_label, None)
        if qc_label is None:
            continue

        rubric = cfg.RUBRIC_PREFIX + str(item["reference_answer"]).strip()

        rows.append({
            "question"          : str(item["question"]).strip(),
            "rubric_descriptor" : rubric,
            "full_rubric"       : rubric,
            "response"          : str(item["student_answer"]).strip(),
            "seb_label"         : raw_label,
            "quality_label"     : qc_label,
            "label_id"          : cfg.LABEL2ID[qc_label],
            "output_type"       : "short_answer",
        })
    return pd.DataFrame(rows)

train_all_df = convert_split(raw["train"],   cfg)
test_ua_df   = convert_split(raw["test_ua"], cfg)
test_uq_df   = convert_split(raw["test_uq"], cfg)
test_ud_df   = convert_split(raw["test_ud"], cfg)

print(f"train (all) : {len(train_all_df):>6,}")
print(f"test_ua     : {len(test_ua_df):>6,}  (Unseen Answers)")
print(f"test_uq     : {len(test_uq_df):>6,}  (Unseen Questions)")
print(f"test_ud     : {len(test_ud_df):>6,}  (Unseen Domains)")

print("\nLabel distribution (train):")
print(train_all_df["quality_label"].value_counts().to_string())
print("\n5-way → 3-way mapping verification (train):")
print(train_all_df.groupby(["seb_label","quality_label"]).size().to_string())


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
split_names = ["Train", "Test UA", "Test UQ", "Test UD"]
dfs         = [train_all_df, test_ua_df, test_uq_df, test_ud_df]
label_order = ["Fully Relevant", "Partially Relevant", "Irrelevant"]
colors      = ["#27ae60", "#f39c12", "#e74c3c"]

for ax, name, df in zip(axes, split_names, dfs):
    counts = [df[df["quality_label"]==l].shape[0] for l in label_order]
    bars   = ax.bar(label_order, counts, color=colors, edgecolor="black")
    ax.set_title(f"{name}  (n={len(df):,})", fontsize=9, fontweight="bold")
    ax.set_ylabel("Count"); ax.tick_params(axis="x", rotation=15)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                str(cnt), ha="center", fontsize=8)

plt.suptitle("SciEntsBank – QualCheck Label Distribution", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


### Step 3 – Text Preprocessing (NLTK + spaCy)

In [ ]:
class TextPreprocessor:
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words("english"))

    def clean_for_bert(self, text):
        return str(text).strip() if not pd.isna(text) else ""

    def deep_clean(self, text):
        if pd.isna(text): return ""
        tokens = nltk.word_tokenize(str(text).lower())
        return " ".join(self.lemmatizer.lemmatize(t)
                         for t in tokens if t.isalnum() and t not in self.stop_words)

prep = TextPreprocessor()

def preprocess_df(df):
    df = df.copy()
    df["clean_question"] = df["question"].apply(prep.clean_for_bert)
    df["clean_response"] = df["response"].apply(prep.clean_for_bert)
    df["tfidf_question"] = df["question"].apply(prep.deep_clean)
    df["tfidf_rubric"]   = df["full_rubric"].apply(prep.deep_clean)
    df["tfidf_response"] = df["response"].apply(prep.deep_clean)
    return df

train_all_df = preprocess_df(train_all_df)
test_ua_df   = preprocess_df(test_ua_df)
test_uq_df   = preprocess_df(test_uq_df)
test_ud_df   = preprocess_df(test_ud_df)

print("✓ Preprocessing complete")
print("\nExample entry:")
print(f"  Q  : {train_all_df['clean_question'].iloc[0]}")
print(f"  R  : {train_all_df['clean_response'].iloc[0]}")
print(f"  Lbl: {train_all_df['quality_label'].iloc[0]}")


### Step 4 – Create Validation Set from Training Data

In [ ]:
# SciEntsBank already has 3 held-out test sets.
# We carve a validation set out of the training split for threshold calibration.
train_df, val_df = train_test_split(
    train_all_df,
    test_size    = cfg.VAL_RATIO,
    stratify     = train_all_df["label_id"],
    random_state = cfg.SEED
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

N = len(train_all_df)
print(f"Train : {len(train_df):>5,}  ({len(train_df)/N*100:.1f}%)")
print(f"Val   : {len(val_df):>5,}  ({len(val_df)/N*100:.1f}%)")
print(f"\nTest splits kept intact:")
print(f"  test_ua : {len(test_ua_df):,}")
print(f"  test_uq : {len(test_uq_df):,}")
print(f"  test_ud : {len(test_ud_df):,}")
print("\nTrain label distribution:")
print(train_df["quality_label"].value_counts().to_string())


## Phase 2 – BERT Encoding & Fine-Tuning

In [ ]:
class QualCheckDataset(Dataset):
    """
    P  →  [CLS] question [SEP] rubric_descriptor [SEP]
    R  →  [CLS] student_answer [SEP]
    """
    def __init__(self, df, tokenizer, max_len):
        self.df  = df.reset_index(drop=True)
        self.tok = tokenizer
        self.ml  = max_len

    def __len__(self): return len(self.df)

    def _enc(self, a, b=None):
        return self.tok.encode_plus(a, b,
            add_special_tokens=True, max_length=self.ml,
            padding="max_length", truncation=True,
            return_attention_mask=True, return_tensors="pt")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pr  = self._enc(row["clean_question"], row["full_rubric"])
        r   = self._enc(row["clean_response"])
        return {
            "pr_ids"  : pr["input_ids"].squeeze(0),
            "pr_mask" : pr["attention_mask"].squeeze(0),
            "r_ids"   : r["input_ids"].squeeze(0),
            "r_mask"  : r["attention_mask"].squeeze(0),
            "label"   : torch.tensor(row["label_id"], dtype=torch.long)
        }


In [ ]:
class QualCheckModel(nn.Module):
    """Siamese BERT with a 4-feature classification head."""
    def __init__(self, bert_name, num_labels=3, dropout=0.3):
        super().__init__()
        self.bert       = BertModel.from_pretrained(bert_name)
        hidden          = self.bert.config.hidden_size
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 4, hidden), nn.Tanh(),
            nn.Dropout(dropout), nn.Linear(hidden, num_labels))

    def encode(self, ids, mask):
        return self.dropout(
            self.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0, :])

    def forward(self, pr_ids, pr_mask, r_ids, r_mask):
        p   = self.encode(pr_ids, pr_mask)
        r   = self.encode(r_ids,  r_mask)
        cos = F.cosine_similarity(p, r, dim=1, eps=1e-8)
        return self.classifier(torch.cat([p, r, torch.abs(p-r), p*r], dim=1)), cos, p, r

    @torch.no_grad()
    def similarity(self, pr_ids, pr_mask, r_ids, r_mask):
        p = self.encode(pr_ids, pr_mask)
        r = self.encode(r_ids,  r_mask)
        return F.cosine_similarity(p, r, dim=1, eps=1e-8)


In [ ]:
cel = nn.CrossEntropyLoss()

def train_epoch(model, loader, opt, sch, device):
    model.train()
    tl = correct = n = 0
    for b in tqdm(loader, desc="  Train", leave=False):
        prid=b["pr_ids"].to(device); prm=b["pr_mask"].to(device)
        rid=b["r_ids"].to(device);   rm=b["r_mask"].to(device)
        lbl=b["label"].to(device)
        opt.zero_grad()
        logits,_,_,_ = model(prid,prm,rid,rm)
        loss = cel(logits, lbl); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()
        tl += loss.item()*lbl.size(0)
        correct += (logits.argmax(1)==lbl).sum().item()
        n += lbl.size(0)
    return tl/n, correct/n

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    tl = correct = n = 0; pa=[]; la=[]
    for b in tqdm(loader, desc="  Eval ", leave=False):
        prid=b["pr_ids"].to(device); prm=b["pr_mask"].to(device)
        rid=b["r_ids"].to(device);   rm=b["r_mask"].to(device)
        lbl=b["label"].to(device)
        logits,_,_,_ = model(prid,prm,rid,rm)
        loss = cel(logits, lbl)
        tl += loss.item()*lbl.size(0)
        p = logits.argmax(1)
        correct += (p==lbl).sum().item(); n += lbl.size(0)
        pa.extend(p.cpu().tolist()); la.extend(lbl.cpu().tolist())
    wf1 = f1_score(la, pa, average="weighted", zero_division=0)
    return tl/n, correct/n, wf1

@torch.no_grad()
def get_sims(model, df, tokenizer, cfg):
    ds  = QualCheckDataset(df, tokenizer, cfg.MAX_LEN)
    ld  = DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
    sims=[]; lbls=[]
    for b in tqdm(ld, desc="  Embed", leave=False):
        cos = model.similarity(
            b["pr_ids"].to(cfg.DEVICE), b["pr_mask"].to(cfg.DEVICE),
            b["r_ids"].to(cfg.DEVICE),  b["r_mask"].to(cfg.DEVICE))
        sims.extend(cos.cpu().tolist()); lbls.extend(b["label"].tolist())
    return np.array(sims), np.array(lbls)


### Run Fine-Tuning

In [ ]:
tokenizer = BertTokenizer.from_pretrained(cfg.BERT_MODEL)
model     = QualCheckModel(cfg.BERT_MODEL, cfg.NUM_LABELS, cfg.DROPOUT).to(cfg.DEVICE)

train_loader = DataLoader(QualCheckDataset(train_df, tokenizer, cfg.MAX_LEN),
                          batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(QualCheckDataset(val_df,   tokenizer, cfg.MAX_LEN),
                          batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

total_steps = len(train_loader) * cfg.MAX_EPOCHS
opt = AdamW(model.parameters(), lr=cfg.LR, eps=1e-8, weight_decay=0.01)
sch = get_linear_schedule_with_warmup(opt, int(total_steps*cfg.WARMUP_RATIO), total_steps)

history = {k:[] for k in ["tr_loss","tr_acc","val_loss","val_acc","val_wf1"]}
best_wf1 = 0.0; no_imp = 0

print(f"Training  |  device={cfg.DEVICE}  |  steps={total_steps}"); print("="*60)
for epoch in range(cfg.MAX_EPOCHS):
    trl, tra        = train_epoch(model, train_loader, opt, sch, cfg.DEVICE)
    vl,  va,  vwf1  = evaluate(model,   val_loader,   cfg.DEVICE)
    for k,v in zip(["tr_loss","tr_acc","val_loss","val_acc","val_wf1"],[trl,tra,vl,va,vwf1]):
        history[k].append(v)
    tag = ""
    if vwf1 > best_wf1:
        best_wf1 = vwf1; torch.save(model.state_dict(), cfg.BEST_CKPT); tag = "  ← best"; no_imp = 0
    else:
        no_imp += 1
    print(f"Ep {epoch+1}  tr_loss={trl:.4f}  tr_acc={tra:.4f}  val_loss={vl:.4f}  val_acc={va:.4f}  val_wf1={vwf1:.4f}{tag}")
    if no_imp >= cfg.PATIENCE:
        print(f"Early stopping at epoch {epoch+1}"); break

print(f"\nBest Val Weighted-F1 : {best_wf1:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ep = range(1, len(history["tr_loss"])+1)
axes[0].plot(ep, history["tr_loss"], label="Train"); axes[0].plot(ep, history["val_loss"], label="Val"); axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(ep, history["tr_acc"],  label="Train"); axes[1].plot(ep, history["val_acc"],  label="Val"); axes[1].set_title("Accuracy"); axes[1].legend()
axes[2].plot(ep, history["val_wf1"], color="green", marker="o")
axes[2].axhline(0.80, color="red", linestyle="--", label="0.80"); axes[2].set_title("Val Weighted F1"); axes[2].legend()
for ax in axes: ax.set_xlabel("Epoch"); ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.suptitle("QualCheck Fine-Tuning – SciEntsBank", fontweight="bold"); plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/training_history.png", dpi=150, bbox_inches="tight"); plt.show()


## Phase 3 – Cosine Similarity Scoring

In [ ]:
model.load_state_dict(torch.load(cfg.BEST_CKPT, map_location=cfg.DEVICE))
model.eval()

print("Extracting cosine similarities …")
val_sims,    val_true    = get_sims(model, val_df,   tokenizer, cfg)
ua_sims,     ua_true     = get_sims(model, test_ua_df, tokenizer, cfg)
uq_sims,     uq_true     = get_sims(model, test_uq_df, tokenizer, cfg)
ud_sims,     ud_true     = get_sims(model, test_ud_df, tokenizer, cfg)

for name, sims in [("Val",val_sims),("Test UA",ua_sims),("Test UQ",uq_sims),("Test UD",ud_sims)]:
    print(f"{name:<10}  min={sims.min():.4f}  max={sims.max():.4f}  mean={sims.mean():.4f}")


In [ ]:
label_names = ["Irrelevant", "Partially Relevant", "Fully Relevant"]
colors_cls  = ["#e74c3c", "#f39c12", "#27ae60"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for lid in range(3):
    m = val_true == lid
    axes[0].hist(val_sims[m], bins=30, alpha=0.6, label=label_names[lid], color=colors_cls[lid])
axes[0].set_title("Cosine Similarity by Class (Val)"); axes[0].set_xlabel("s"); axes[0].legend()

bp = axes[1].boxplot([val_sims[val_true==i] for i in range(3)],
                     labels=label_names, patch_artist=True)
for patch, c in zip(bp["boxes"], colors_cls): patch.set_facecolor(c); patch.set_alpha(0.7)
axes[1].set_title("Boxplot by Class (Val)"); axes[1].set_ylabel("Cosine Similarity  s")
plt.suptitle("SciEntsBank – Semantic Alignment Distribution", fontweight="bold"); plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/similarity_distribution.png", dpi=150, bbox_inches="tight"); plt.show()


## Phase 4 – Threshold Calibration (Grid Search on Val)

In [ ]:
def classify(sims, t1, t2):
    p = np.zeros(len(sims), dtype=int)
    p[sims >= t1]                  = 2
    p[(sims >= t2) & (sims < t1)] = 1
    return p

def grid_search(sims, true_labels, step=0.01):
    cands = np.arange(float(sims.min()), float(sims.max()), step)
    best  = dict(t1=0.8, t2=0.5, score=0.0)
    for t1 in cands:
        for t2 in cands:
            if t2 >= t1: continue
            pr  = classify(sims, t1, t2)
            acc = accuracy_score(true_labels, pr)
            wf1 = f1_score(true_labels, pr, average="weighted", zero_division=0)
            s   = (acc + wf1) / 2
            if s > best["score"]:
                best.update(t1=t1, t2=t2, score=s)
    return best

print("Running grid search on validation set …")
res = grid_search(val_sims, val_true, cfg.THRESH_STEP)
θ1  = res["t1"]; θ2 = res["t2"]

print(f"\n{'='*50}")
print(f" θ₁ (Fully Relevant   boundary) : {θ1:.4f}")
print(f" θ₂ (Irrelevant       boundary) : {θ2:.4f}")
print(f" Combined score                 : {res['score']:.4f}")
print(f"\n s ≥ {θ1:.4f}               → Fully Relevant")
print(f" {θ2:.4f} ≤ s < {θ1:.4f}  → Partially Relevant")
print(f" s <  {θ2:.4f}               → Irrelevant")


## Phase 5 – Evaluation on All Three Test Splits

SciEntsBank has three distinct generalisation challenges:
- **test_ua** – same questions, same domains, new student answers
- **test_uq** – same domains, new questions (never seen at training)
- **test_ud** – entirely new science domains (hardest)

In [ ]:
def evaluate_split(sims, true_labels, θ1, θ2, split_name):
    preds = classify(sims, θ1, θ2)
    acc   = accuracy_score(true_labels, preds)
    prec  = precision_score(true_labels, preds, average="weighted", zero_division=0)
    rec   = recall_score(true_labels, preds,    average="weighted", zero_division=0)
    f1m   = f1_score(true_labels, preds, average="macro",    zero_division=0)
    f1w   = f1_score(true_labels, preds, average="weighted", zero_division=0)
    norm  = true_labels / (cfg.NUM_LABELS - 1)
    mae   = np.mean(np.abs(sims - norm))
    rmse  = np.sqrt(np.mean((sims - norm)**2))
    r, pr = pearsonr(sims, norm) if len(sims) > 2 else (float("nan"), float("nan"))

    print(f"\n{'='*55}")
    print(f" {split_name}  (n={len(true_labels):,})")
    print(f"{'='*55}")
    print(f" Accuracy       : {acc:.4f}")
    print(f" Precision (W)  : {prec:.4f}")
    print(f" Recall    (W)  : {rec:.4f}")
    print(f" F1 (macro)     : {f1m:.4f}")
    print(f" Weighted F1    : {f1w:.4f}  ({'✓' if f1w>=0.80 else '✗'} ≥ 0.80)")
    print(f" MAE            : {mae:.4f}")
    print(f" RMSE           : {rmse:.4f}")
    print(f" Pearson r      : {r:.4f}  ({'✓' if r>=0.70 else '✗'} ≥ 0.70)")
    print(f"\nClassification Report:")
    print(classification_report(true_labels, preds, target_names=label_names))
    return dict(split=split_name, n=len(true_labels),
                acc=acc, prec=prec, rec=rec, f1_macro=f1m,
                weighted_f1=f1w, mae=mae, rmse=rmse, pearson_r=r), preds

results_ua, preds_ua = evaluate_split(ua_sims, ua_true, θ1, θ2, "Test UA – Unseen Answers")
results_uq, preds_uq = evaluate_split(uq_sims, uq_true, θ1, θ2, "Test UQ – Unseen Questions")
results_ud, preds_ud = evaluate_split(ud_sims, ud_true, θ1, θ2, "Test UD – Unseen Domains")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cls_names = ["Irrelevant", "Partially\nRelevant", "Fully\nRelevant"]

for ax, (name, true, pred) in zip(axes, [
        ("Test UA", ua_true, preds_ua),
        ("Test UQ", uq_true, preds_uq),
        ("Test UD", ud_true, preds_ud)]):
    cm = confusion_matrix(true, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=cls_names, yticklabels=cls_names,
                linewidths=0.5, ax=ax)
    ax.set_title(f"Confusion Matrix – {name}", fontweight="bold")
    ax.set_ylabel("True"); ax.set_xlabel("Predicted")

plt.suptitle("QualCheck Confusion Matrices – SciEntsBank", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()


### Cross-Split Performance Summary

In [ ]:
summary_df = pd.DataFrame([results_ua, results_uq, results_ud])
summary_df.to_csv(f"{cfg.OUTPUT_DIR}/split_results_summary.csv", index=False)

print("Cross-Split Summary")
print(summary_df[["split","n","acc","weighted_f1","f1_macro","pearson_r","mae"]].to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics  = ["weighted_f1", "acc", "pearson_r"]
mtitles  = ["Weighted F1",  "Accuracy", "Pearson r"]
split_l  = ["Test UA", "Test UQ", "Test UD"]
bar_cols = ["#5dade2", "#a569bd", "#48c9b0"]
thresholds = [0.80, 0.80, 0.70]

for ax, met, title, thr in zip(axes, metrics, mtitles, thresholds):
    vals = summary_df[met].tolist()
    bars = ax.bar(split_l, vals, color=bar_cols, edgecolor="black", width=0.5)
    ax.set_ylim(0, 1.1); ax.axhline(thr, color="red", linestyle="--", lw=1.2, label=f"Threshold {thr}")
    ax.set_title(f"{title} by Split", fontweight="bold"); ax.legend(fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f"{v:.4f}",
                ha="center", fontweight="bold", fontsize=9)

plt.suptitle("QualCheck – SciEntsBank Cross-Split Performance", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/cross_split_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


### Baseline Comparisons (Test UA)

In [ ]:
# Use test_ua as the primary comparison split
print("--- Baseline 1: TF-IDF Cosine Similarity ---")
all_tfidf = (list(train_df["tfidf_question"] + " " + train_df["tfidf_rubric"]) +
             list(train_df["tfidf_response"]))
tv = TfidfVectorizer(max_features=10_000, ngram_range=(1,2)); tv.fit(all_tfidf)

def tfidf_sims(df_sp):
    pr = tv.transform(df_sp["tfidf_question"] + " " + df_sp["tfidf_rubric"])
    r  = tv.transform(df_sp["tfidf_response"])
    return np.array([tfidf_cos(pr[i], r[i])[0,0] for i in range(len(df_sp))])

bl1_val = tfidf_sims(val_df);    bl1_ua  = tfidf_sims(test_ua_df)
bl1_res = grid_search(bl1_val, val_true, 0.01)
bl1_p   = classify(bl1_ua, bl1_res["t1"], bl1_res["t2"])
bl1_wf1 = f1_score(ua_true, bl1_p, average="weighted", zero_division=0)
bl1_acc = accuracy_score(ua_true, bl1_p)
bl1_r,_ = pearsonr(bl1_ua, ua_true/(cfg.NUM_LABELS-1))
bl1_mae = np.mean(np.abs(bl1_ua - ua_true/(cfg.NUM_LABELS-1)))
print(f"  Acc={bl1_acc:.4f}  WF1={bl1_wf1:.4f}  r={bl1_r:.4f}  MAE={bl1_mae:.4f}")

print("\n--- Baseline 2: BERT without rubric (question only) ---")
@torch.no_grad()
def bert_no_rubric(model, df_sp, tokenizer, cfg):
    sims = []
    model.eval()
    for _, row in tqdm(df_sp.iterrows(), total=len(df_sp), desc="  BL2", leave=False):
        q = tokenizer.encode_plus(row["clean_question"],
              add_special_tokens=True, max_length=cfg.MAX_LEN,
              padding="max_length", truncation=True,
              return_attention_mask=True, return_tensors="pt")
        r = tokenizer.encode_plus(row["clean_response"],
              add_special_tokens=True, max_length=cfg.MAX_LEN,
              padding="max_length", truncation=True,
              return_attention_mask=True, return_tensors="pt")
        sims.append(model.similarity(
            q["input_ids"].to(cfg.DEVICE), q["attention_mask"].to(cfg.DEVICE),
            r["input_ids"].to(cfg.DEVICE), r["attention_mask"].to(cfg.DEVICE)).item())
    return np.array(sims)

bl2_val = bert_no_rubric(model, val_df,     tokenizer, cfg)
bl2_ua  = bert_no_rubric(model, test_ua_df, tokenizer, cfg)
bl2_res = grid_search(bl2_val, val_true, 0.01)
bl2_p   = classify(bl2_ua, bl2_res["t1"], bl2_res["t2"])
bl2_wf1 = f1_score(ua_true, bl2_p, average="weighted", zero_division=0)
bl2_acc = accuracy_score(ua_true, bl2_p)
bl2_r,_ = pearsonr(bl2_ua, ua_true/(cfg.NUM_LABELS-1))
bl2_mae = np.mean(np.abs(bl2_ua - ua_true/(cfg.NUM_LABELS-1)))
print(f"  Acc={bl2_acc:.4f}  WF1={bl2_wf1:.4f}  r={bl2_r:.4f}  MAE={bl2_mae:.4f}")

qc_ua_wf1 = results_ua["weighted_f1"]; qc_ua_acc = results_ua["acc"]
qc_ua_r   = results_ua["pearson_r"];   qc_ua_mae = results_ua["mae"]

comp = pd.DataFrame({
    "System"      : ["BL1 – TF-IDF", "BL2 – BERT (no rubric)", "QualCheck (Ours)"],
    "Accuracy"    : [bl1_acc,  bl2_acc,  qc_ua_acc],
    "Weighted F1" : [bl1_wf1,  bl2_wf1,  qc_ua_wf1],
    "Pearson r"   : [bl1_r,    bl2_r,    qc_ua_r],
    "MAE"         : [bl1_mae,  bl2_mae,  qc_ua_mae],
    "ΔWF1 vs BL1" : [0.0, round(bl2_wf1-bl1_wf1,4), round(qc_ua_wf1-bl1_wf1,4)],
})
comp.to_csv(f"{cfg.OUTPUT_DIR}/baseline_comparison.csv", index=False)
print("\nBaseline Comparison (Test UA):")
print(comp.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sl = ["BL1\n(TF-IDF)", "BL2\n(BERT)", "QualCheck"]
bc = ["#bdc3c7", "#f0b27a", "#82e0aa"]
for ax, met in zip(axes, ["Weighted F1", "Accuracy"]):
    vals = comp[met].tolist()
    bars = ax.bar(sl, vals, color=bc, edgecolor="black", width=0.5)
    ax.set_ylim(0, 1.1); ax.axhline(0.80, color="red", linestyle="--", lw=1.2, label="0.80")
    ax.set_title(f"{met} – Test UA", fontweight="bold"); ax.legend()
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f"{v:.4f}",
                ha="center", fontweight="bold", fontsize=9)
plt.suptitle("System Comparison – SciEntsBank Test UA", fontweight="bold"); plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/baseline_comparison.png", dpi=150, bbox_inches="tight"); plt.show()


### Statistical Tests

In [ ]:
# 1. Pearson r per split
print("1. Score-Level Agreement (Pearson r)")
for name, sims, true in [("Test UA", ua_sims, ua_true),
                          ("Test UQ", uq_sims, uq_true),
                          ("Test UD", ud_sims, ud_true)]:
    r, p = pearsonr(sims, true/(cfg.NUM_LABELS-1))
    print(f"   {name}: r={r:.4f}  p={p:.4f}  {'✓' if r>=0.70 else '✗'}")

# 2. One-way ANOVA across splits
print("\n2. One-Way ANOVA – consistency across test splits")
F, pv = f_oneway(ua_sims, uq_sims, ud_sims)
print(f"   F={F:.4f}  p={pv:.4f}")
if pv > 0.05:
    print("   Non-significant (p>0.05): consistent across splits ✓")
else:
    print("   Significant (p≤0.05): Tukey HSD:")
    all_s = np.concatenate([ua_sims, uq_sims, ud_sims])
    all_g = ["UA"]*len(ua_sims) + ["UQ"]*len(uq_sims) + ["UD"]*len(ud_sims)
    print(pairwise_tukeyhsd(all_s, all_g, alpha=0.05))

# 3. Descriptive stats (using test_ua as reference)
print("\n3. Descriptive Statistics – Test UA (cosine similarity by class)")
print(f"{'Class':<22} {'Mean':>6} {'Median':>8} {'Std':>6} {'IQR':>6}")
print("-"*50)
for lid, lname in cfg.ID2LABEL.items():
    m = ua_true == lid
    if m.any():
        s = ua_sims[m]
        iqr = np.percentile(s,75)-np.percentile(s,25)
        print(f"{lname:<22} {s.mean():>6.4f} {np.median(s):>8.4f} {s.std():>6.4f} {iqr:>6.4f}")


## Save Final Model & Predictions

In [ ]:
torch.save({
    "model_state_dict" : model.state_dict(),
    "theta1"           : θ1,
    "theta2"           : θ2,
    "bert_model"       : cfg.BERT_MODEL,
    "max_len"          : cfg.MAX_LEN,
    "label_map"        : cfg.LABEL2ID,
    "seb_to_qc"        : cfg.SEB_TO_QC,
    "rubric_prefix"    : cfg.RUBRIC_PREFIX,
}, cfg.FINAL_MODEL)

# Save predictions for each test split
for name, df_sp, sims, true, preds in [
        ("test_ua", test_ua_df, ua_sims, ua_true, preds_ua),
        ("test_uq", test_uq_df, uq_sims, uq_true, preds_uq),
        ("test_ud", test_ud_df, ud_sims, ud_true, preds_ud)]:
    out = df_sp[["question","rubric_descriptor","response","quality_label"]].copy()
    out["cosine_similarity"] = sims
    out["predicted_label"]   = [cfg.ID2LABEL[p] for p in preds]
    out["correct"]           = (true == preds)
    out.to_csv(f"{cfg.OUTPUT_DIR}/predictions_{name}.csv", index=False)

print("="*60)
print(" QUALCHECK – SCIENTSBANK FINAL RESULTS")
print("="*60)
print(f" θ₁ = {θ1:.4f}   θ₂ = {θ2:.4f}")
print(f" {'Split':<12} {'WF1':>6} {'Acc':>6} {'r':>6} {'MAE':>6}")
print(" "+"-"*35)
for r in [results_ua, results_uq, results_ud]:
    vf = r["weighted_f1"]; va = r["acc"]; vr = r["pearson_r"]; vm = r["mae"]
    ok = "✓" if vf>=0.80 else "✗"
    print(f" {r['split'].split('–')[0].strip():<12} {vf:>6.4f}{ok} {va:>6.4f} {vr:>6.4f} {vm:>6.4f}")
print("="*60)
print("✓ QualCheck – SciEntsBank pipeline complete!")
